In [1]:
%matplotlib inline
import os
os.environ["OMP_NUM_THREADS"] = '3'
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.metrics import r2_score as r2
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn import set_config


NbaOG_csv = 'NBA_2024_per_game(03-01-2024).csv'
NbaOG = pd.read_csv(NbaOG_csv)
#NbaOG.dtypes

FileNotFoundError: [Errno 2] No such file or directory: 'NBA_2024_per_game(03-01-2024).csv'

#### Data Cleaning and Prepartion ####

In [ ]:
#Check for missing values 

NbaOG.isnull().any()
NbaOG.isnull().sum()

AvgNba = NbaOG[['PTS', 'FG%' , '3P%' , '2P%', 'eFG%', 'FT%']].mean()

# #checking work
#print(AvgNba.isnull().any())

NbaOG[['PTS', 'FG%' , '3P%' , '2P%', 'eFG%', 'FT%']].fillna(value=AvgNba)

In [ ]:
#list = (NbaOG['Pos'].value_counts())
#print(list)

#drop the POS less than 1 C-PF, SF-PF, SG-PG
NBA = NbaOG.groupby('Pos').filter(lambda x : len(x)>1)
#https://stackoverflow.com/questions/74931345/how-to-drop-rows-from-a-dataset-where-the-value-counts-are-low
list = (NBA['Pos'].value_counts())

# #drop columns Age 
NBA = NBA.drop(['Age'], axis = 1)
display(NBA)


# group by Player
NBATeams= NBA.groupby('Tm')[['PTS','FG%' , '3P%' , '2P%', 'eFG%', 'FT%', 'AST', 'STL', 'BLK' ]].mean()

NBATeams = NBATeams.round(2)
NBATeams = NBATeams.sort_values(by = ["PTS"], ascending = False)


# top 5 players by 3P%
NBA3P = NBATeams.sort_values(by = ["3P%"], ascending = False)
NBA3P_5 = NBATeams.iloc[: 5]

#top 5 players 2P%

NBA2P = NBATeams.sort_values(by = ["2P%"], ascending = False)
NBA2P_5 = NBATeams.iloc[6: 11]

NBAall = pd.concat([NBA2P_5, NBA3P_5], axis = 0)
#https://www.geeksforgeeks.org/python/merge-two-dataframes-with-same-column-names/


for col in NBAall.columns: 
    NBAall[col] = NBAall[col].fillna(NBAall[col].mean())
#https://youtu.be/t0_vyEs1uOs?si=CAnwn1MFrF3gpdFL

NBAall = NBAall.round(2)
NBAall = NBAall.sort_values(by = ["PTS"], ascending = False)

NBATop = NBAall.drop_duplicates( keep = 'first')

#https://www.geeksforgeeks.org/python/count-the-number-of-rows-and-columns-of-a-pandas-dataframe/
NBATop = NBATop.reset_index()

In [ ]:
#add a column for conference - east or west 

#plugged in table to google and returned the conference for each team 
# Summary Table
# Team Code	Team Name	Conference
# MIA	Miami Heat	East
# PHO	Phoenix Suns	West
# BRK	Brooklyn Nets	East
# CLE	Cleveland Cavaliers	East
# CHO	Charlotte Hornets	East
# IND	Indiana Pacers	East
# NOP	New Orleans Pelicans	West
# TOR	Toronto Raptors	East
# DET	Detroit Pistons	East
# MEM	Memphis Grizzlies	West

def classify(x):
    if x == 'MEM':
        return 'West'
    elif x == 'NOP':
        return 'West'
    elif x == 'PHO':
        return 'West'
    else:
        return 'East'
NBATop['Conference'] = NBATop['Tm'].apply(classify)
# https://www.geeksforgeeks.org/python/using-apply-in-pandas-lambda-functions-with-multiple-if-statements/
display(NBATop)

### Exploratory Data Analysis Tasks ###

#### Summary Statistics and Univariate Analysis ####

In [ ]:
NBATop.shape
#https://www.geeksforgeeks.org/python/count-the-number-of-rows-and-columns-of-a-pandas-dataframe/


In [ ]:
import statistics
#calculate descriptive stats
#provide a description of variables using summary stats
display(NBATop.describe())

#mode
modes = NBATop[['PTS','FG%' , '3P%' , '2P%', 'eFG%', 'FT%', 'AST', 'STL', 'BLK']].mode(dropna = True)
print("The following is the Mode:")
print(modes)
print()
#variance 
print("The following is the Variance:")
print(np.var(NBATop[['PTS','FG%' , '3P%' , '2P%', 'eFG%', 'FT%', 'AST', 'STL', 'BLK']] ,axis = 0))



In [ ]:
#skewness
skew = NBATop[['PTS','FG%' , '3P%' , '2P%', 'eFG%', 'FT%', 'AST', 'STL', 'BLK']].skew(axis = 0, skipna = True)
print('The following is the Skeweness:')
print(skew)
#kurtosis 
kurt = NBATop[['PTS','FG%' , '3P%' , '2P%', 'eFG%', 'FT%', 'AST', 'STL', 'BLK']].kurtosis(axis = 0 , skipna = True)
print("The following is the Kurtosis:")
print(kurt)

In [ ]:
#visualizations of Descriptive Stats 

fig, axes = plt.subplots(1, 6, figsize = (12,6))

sns.histplot(NBATop["2P%"], kde = True, ax = axes[0])

sns.histplot(NBATop["3P%"], kde = True, ax = axes [1])

sns.histplot(NBATop['FG%'], kde = True, ax = axes[2])

sns.histplot(NBATop['Conference'], ax = axes[3])

sns.histplot(kurt, kde = True, ax = axes[4])

sns.histplot(skew, kde = True, ax = axes [5])


plt.tight_layout()
plt.show()

In [ ]:

threemean = NBATop['3P%'].mean()
twomean = NBATop['2P%'].mean()

fig, axes = plt.subplots(1, 2, figsize = (12,6))
markers = {"3P%" : "s" , "2P%" : "X"}
sns.scatterplot(NBATop, x = 'Tm' ,y = '3P%', hue = 'Conference', markers = markers, ax= axes[0])
axes[0].set_ylim([0 , .75])
axes[0].axhline(y=threemean)
sns.scatterplot(NBATop, x = 'Tm' ,y = '2P%', hue = 'Conference', markers = markers, ax= axes[1])
axes[1].set_ylim([0 , .75])
axes[1].axhline(y=twomean)

#### Conduct Bivariate Analysis ####

In [ ]:
#Calculate Covariance and Correlation 

cov = NBATop[['PTS','FG%' , '3P%' , '2P%', 'eFG%', 'FT%', 'AST', 'STL', 'BLK']].cov(min_periods = 1)
print("The CoVariance is displayed on the following dataframe:" )  
display(cov)
#Pearson for continuous values

numcorr = NBATop[['PTS','FG%' , '3P%' , '2P%', 'eFG%', 'FT%', 'AST', 'STL', 'BLK']].corr(method = "pearson")
print("The Pearson method is used to display on the following correlation dataframe:" )
display(numcorr)

#Spearman rank for categorical

dummies  = (pd.get_dummies(NBATop['Tm'], dummy_na = True, dtype = 'int'))
nbacorr = pd.concat([dummies, NBAall], axis = 1)
# display(nbacorr)
Tmcorr = dummies.corr()
print("The Team correlation based on categorical data:")
display(Tmcorr)

In [ ]:
#plot CoVariance and Correlation

fig, axes = plt.subplots(1, 2, figsize = (12,6))

sns.heatmap(cov, annot = True, fmt = '.0g', ax = axes [0])
sns.heatmap(numcorr, annot = True, fmt = '.1g', ax = axes[1])

plt.tight_layout()
plt.show()

#### Test Statistics ####

In [ ]:
#Developer a Hyptohesis 

#T-Test 
NBAtop5 = NBATop.sort_values(by = ["PTS"], ascending = False)
NBAtop5 = NBATop.iloc[: 5]
NBAtop5 = NBAtop5.drop(['Tm','Conference'], axis = 1)
display(NBAtop5)
NBAbot5 = NBATop.sort_values(by = ["PTS"], ascending = False)
NBAbot5 = NBATop.iloc[5: 10]
NBAbot5 = NBAbot5.drop(['Tm', 'Conference'], axis = 1)
display(NBAbot5)

#Explain test stats results, interpet p-values, draw conclusions

In [ ]:
#perform t-test

from scipy.stats import ttest_ind
ttest = ttest_ind(NBAtop5['3P%'], NBAbot5['3P%'])
display(ttest)
#https://www.statology.org/pandas-t-test/

##### #how does 3 pointers impact a team's success #####

The top ten teams were compared by their three-point percentages. The information confirms the hypothesis is not rejected because the p-value is .22%. There is a significant in 3 pointers and the success of a team.  

#### Regression Linear Regression ####

In [ ]:
y = NBATop.drop(columns = (['Tm', 'Conference']))

X = NBATop[['PTS','FG%' , '3P%' , '2P%', 'FT%']]



X_train , X_test , y_train, y_test = train_test_split(X, y, test_size = .4,  random_state = 42)

# print(X_train.shape)
# print(X_test.shape)
# print(y_train.shape)
# print(y_test.shape)
# print(X.shape)
# print(y.shape)


lr = LinearRegression()
lr.fit(X,y)

print("The following is in order for 'PTS', 'FG%', '3P%', '2P%', 'FT%'.")
print("The Coefficient is")
display(lr.coef_[0])
print("")
print("The model intercept is")
display(lr.intercept_)

In [ ]:
#model predictions
y_pred = lr.predict(X_test)

y_pred_df = pd.DataFrame(y_pred)
y_pred_t = y_pred_df.T
display(y_pred_t)


#Evaulate Model 
abos = mae(y_test, y_pred)
sqr = mse(y_test, y_pred)
r21 = r2(y_test,y_pred)

print(f"The mean absolute error is {abos:2f}""." )
print(f"The mean squared error is {sqr:2f}"".")
print(f"The r2_score is {r21:2f}"".")

# https://medium.com/@heyamit10/how-to-perform-linear-regression-using-pandas-scikit-learn-9fcfa6085fb0

#### Classification KNN ####

In [ ]:
NBAplay = NbaOG.fillna(value = 0, inplace = True)

#NBAplay = NbaOG.groupby('Pos')[['PTS','AST', 'BLK','STL','FTA']].mean()
NBAplay = NBAplay.reset_index()


#https://stackoverflow.com/questions/60102928/pandas-fillna-only-numeric-int-or-float-columns

sns.barplot(NBAplay, x = 'Pos' , y = 'FTA', hue = 'Pos')

FTAmean  = NBAplay['FTA'].mean()
plt.axhline(y=FTAmean)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score


NBAPF = NBAplay.drop(columns = ['Player', 'Tm', 'Age'])

#get_dummies
NBA1 = pd.get_dummies(NBAPF['Pos'], drop_first = True, dtype = int)
NBAC = pd.concat([NBAPF,NBA1], axis = 1) 

NBC = NBAC.dropna()

X = NBC.drop('Pos', axis = 1)
y = NBC.FTA
y = y.astype(int)

X_train , X_test , y_train, y_test = train_test_split(X, y, test_size = .4,  random_state = 42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print(X.shape)
print(y.shape)

scale = MinMaxScaler()


knn = KNeighborsClassifier(n_neighbors = 5, n_jobs = -1)

Pipes = Pipeline([("MinMaxStandardizer", scale), ("knn", knn)])

Pipes.fit(X_train, y_train)


In [ ]:
#create a space of candidate values 

search = [{"knn__n_neighbors":[1,2,3,4,5]}]
#create a grid scearch 

classi = GridSearchCV(
    Pipes, search, cv = 3, verbose = 0).fit(X_train, y_train)

classi.best_estimator_.get_params()["knn__n_neighbors"]

best_model = classi.best_estimator_

y_pred_grid = best_model.predict(X_test)

grid_acc = accuracy_score(y_test, y_pred_grid)
print(grid_acc)

#KNN assist from textbook "Machine Learning with Python Cookbook"

In [ ]:
from sklearn.metrics import classification_report as cr

print(cr(y_test, y_pred_grid))

#### Advanced Analysis Techniques ####

#### Clustering ####

In [ ]:
from sklearn.cluster import KMeans


kmeans_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('kmeans', KMeans(5, random_state = 0))
])

kmeansdata = NBAC[['3P','3PA']].dropna()

kmeans_pipe.fit(kmeansdata)

#https://stackoverflow.com/questions/69596239/how-to-avoid-memory-leak-when-dealing-with-kmeans-for-example-in-this-code-i-am
#added import os to control a memory leak 

In [ ]:
fig, ax = plt.subplots(1,1, figsize = (10,10))
sns.scatterplot(
    x=kmeansdata['3P'],
    y=kmeansdata['3PA'],
    hue=kmeans_pipe.predict(kmeansdata),
    ax=ax, palette = 'Accent'
)


ax.set_title('Cluster of 3 pointer average compared to number of 3 pointers taken')
#obstacle - could not get a legend to specify colors - 
#I wanted to show the names or the teams and where they stand 

In [ ]:
from sklearn.metrics import silhouette_score
silhouette_score(
    kmeansdata, kmeans_pipe.predict(kmeansdata)
)

In [ ]:
from sklearn.metrics import calinski_harabasz_score as chz

#higher score == better defined ratios
chz(kmeansdata, kmeans_pipe.predict(kmeansdata))